# Fruit Recognition using Support Vector Machine based on Deep Features

**Author:** Deep Learning & Vision Pipeline  
**Dataset:** Kaggle - Fruit Classification (10 class)  
**Methodology:** Pretrained CNN Transfer Learning (No fine-tuning) + Support Vector Machine (SVM) Classification Head

---

## 1. Project Overview & Motivation

Classical computer vision pipelines traditionally relied on hand-crafted features (such as SIFT, HOG, or Color Histograms) fed into classical classifiers like Support Vector Machines (SVM). While Convolutional Neural Networks (CNNs) achieve state-of-the-art representation learning, training deep neural networks from scratch on moderate-sized datasets often leads to severe overfitting and requires excessive computational resources.

This project implements an **optimal hybrid approach**:
1. **Feature Extraction:** We leverage deep representations learned by a state-of-the-art CNN backbone (e.g., **ResNet50**, **MobileNetV2**) pretrained on the 1.4-million-image ImageNet dataset. The network acts strictly as a frozen feature extractor without fine-tuning.
2. **Global Average Pooling:** By removing the top classification head and extracting the global average pooled feature vectors (e.g., 2048-D for ResNet50), we obtain rich, high-level semantic embeddings.
3. **SVM Decision Boundary:** We feed these standardized deep features directly into a **Support Vector Machine (SVM)**. SVMs excel at finding the maximum-margin separating hyperplane in high-dimensional spaces, providing robust generalization and avoiding the convergence pitfalls of end-to-end backpropagation.


In [ ]:
# Setup project path & imports
import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torchvision.transforms as transforms
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from src.data_loader import (
    scan_and_collect_images,
    create_train_val_test_splits,
    CANONICAL_CLASSES,
    CLASS_TO_IDX,
    IDX_TO_CLASS
)
from src.feature_extractor import DeepFeatureExtractor, load_cached_features

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print("Environment initialized successfully!")
print(f"Project Root: {PROJECT_ROOT}")


## 2. Exploratory Data Analysis (EDA)

Let's inspect the dataset directory, class distributions, and examine sample images across all 10 canonical fruit categories:
- Apple
- Avocado
- Banana
- Cherry
- Kiwi
- Mango
- Orange
- Pineapple
- Strawberry
- Watermelon


In [ ]:
raw_dir = os.path.join(PROJECT_ROOT, "data", "raw")
paths, labels, counts = scan_and_collect_images(raw_dir)

print(f"Total labeled images discovered: {len(paths)}")
print("\nCanonical Class Counts:")
for cls in CANONICAL_CLASSES:
    print(f"  {cls:12s}: {counts[cls]} images")

# Stratified 70/15/15 split
(t_p, t_l), (v_p, v_l), (te_p, te_l) = create_train_val_test_splits(paths, labels, random_seed=42)
print(f"\nSplits:")
print(f"  Train:      {len(t_p)} ({len(t_p)/len(paths)*100:.1f}%)")
print(f"  Validation: {len(v_p)} ({len(v_p)/len(paths)*100:.1f}%)")
print(f"  Test:       {len(te_p)} ({len(te_p)/len(paths)*100:.1f}%)")

# Plot Class Balance
plt.figure(figsize=(10, 4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()), palette='viridis')
plt.title("Fruit Category Distribution Across Dataset", fontsize=13, fontweight='bold')
plt.xlabel("Fruit Category", fontsize=11)
plt.ylabel("Number of Images", fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Display sample images from each of the 10 classes
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

sample_images = {}
for p, l in zip(paths, labels):
    c_name = IDX_TO_CLASS[l]
    if c_name not in sample_images:
        sample_images[c_name] = p
    if len(sample_images) == 10:
        break

for i, cls in enumerate(CANONICAL_CLASSES):
    img_path = sample_images[cls]
    img = Image.open(img_path).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(cls.capitalize(), fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.suptitle("Sample Images from the 10 Fruit Classes", fontsize=15, y=0.98, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Deep Feature Extraction

We utilize a pretrained **ResNet50** CNN backbone trained on ImageNet.
- **Frozen Weights:** All convolutional and residual blocks remain fixed (`requires_grad = False`).
- **Penultimate Layer:** The final 1000-class dense classification layer is removed.
- **Global Average Pooling (GAP):** The spatial 7x7 feature maps from the final residual stage are pooled into a compact, semantically rich **2048-dimensional** vector per image.


In [ ]:
# Load pre-extracted deep features
feat_path = os.path.join(PROJECT_ROOT, "data", "processed", "features_resnet50.npz")
data = load_cached_features(feat_path)

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
class_names = [str(c) for c in data['class_names']]

print(f"Deep Features Loaded Successfully!")
print(f"Backbone:           {data['backbone']}")
print(f"Feature Dimension:  {data['feature_dim']}")
print(f"X_train shape:      {X_train.shape}")
print(f"X_val shape:        {X_val.shape}")
print(f"X_test shape:       {X_test.shape}")


## 4. Deep Feature Space Visualization (PCA & t-SNE)

Before training the SVM, let's explore whether the extracted 2048-D deep representations already form distinct geometric clusters corresponding to fruit classes. We reduce the feature space to 2 dimensions using **Principal Component Analysis (PCA)** and **t-Distributed Stochastic Neighbor Embedding (t-SNE)**.


In [ ]:
# Standardize features prior to dimensionality reduction
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. PCA 2D Projection
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)

# 2. t-SNE 2D Projection (on a subset of 1000 samples for fast interactive rendering)
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_scaled), size=min(1000, len(X_train_scaled)), replace=False)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter_without_progress=300)
X_sample_tsne = tsne.fit_transform(X_train_scaled[sample_idx])
y_sample = y_train[sample_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot PCA
scatter1 = sns.scatterplot(
    x=X_train_pca[:, 0], y=X_train_pca[:, 1],
    hue=[class_names[i] for i in y_train],
    palette='tab10', alpha=0.7, s=25, ax=ax1
)
ax1.set_title(f"PCA 2D Projection (Explained Var: {np.sum(pca.explained_variance_ratio_)*100:.1f}%)", fontsize=12, fontweight='bold')
ax1.set_xlabel("PC 1")
ax1.set_ylabel("PC 2")
ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

# Plot t-SNE
scatter2 = sns.scatterplot(
    x=X_sample_tsne[:, 0], y=X_sample_tsne[:, 1],
    hue=[class_names[i] for i in y_sample],
    palette='tab10', alpha=0.8, s=35, ax=ax2
)
ax2.set_title("t-SNE 2D Manifold Visualization of Deep Features", fontsize=12, fontweight='bold')
ax2.set_xlabel("t-SNE Dim 1")
ax2.set_ylabel("t-SNE Dim 2")
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)

plt.tight_layout()
plt.show()


## 5. Support Vector Machine (SVM) Classification & Grid Search

We train a multi-class Support Vector Machine directly on the standardized deep features.
We compare:
- **Linear Kernel:** Fast, highly effective when feature dimensionality is high (2048-D) relative to sample size.
- **RBF (Radial Basis Function) Kernel:** Maps deep features into an infinite-dimensional reproducing kernel Hilbert space, capturing subtle non-linear boundaries.


In [ ]:
# Define hyperparameter grid
param_grid = [
    {'kernel': ['linear'], 'C': [0.01, 0.1, 1.0, 10.0]},
    {'kernel': ['rbf'], 'C': [0.1, 1.0, 10.0], 'gamma': ['scale', 0.001, 0.01]}
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV for SVM...")
grid.fit(X_train_scaled, y_train)

print(f"\nBest Cross-Validation Accuracy: {grid.best_score_*100:.2f}%")
print(f"Best Parameters: {grid.best_params_}")

best_svm = grid.best_estimator_


## 6. Evaluation on Held-Out Test Set

Let's evaluate the tuned SVM classifier on the strictly held-out test split (499 images) never seen during training or cross-validation.


In [ ]:
y_pred = best_svm.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='macro')

print(f"Test Accuracy:  {test_acc * 100:.2f}%")
print(f"Macro F1-Score: {test_f1:.4f}")
print("\n" + "="*60)
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))
print("="*60)

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.1%',
    cmap='Blues',
    xticklabels=[c.capitalize() for c in class_names],
    yticklabels=[c.capitalize() for c in class_names],
    cbar_kws={'label': 'Recall'}
)
plt.title(f"Test Confusion Matrix — SVM on ResNet50 Deep Features\nAccuracy: {test_acc*100:.2f}% | Macro F1: {test_f1:.4f}", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Predicted Class", fontsize=11)
plt.ylabel("True Class", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Sample Inference Demo

Finally, we demonstrate end-to-end inference on random test images, showing the predicted fruit label alongside the top predicted category confidence percentages.


In [ ]:
# Select 3 random test samples
paths_test = data['paths_test']
sample_indices = np.random.choice(len(X_test), size=3, replace=False)

fig, axes = plt.subplots(3, 2, figsize=(12, 9))

for row, idx in enumerate(sample_indices):
    img_path = paths_test[idx]
    true_label = class_names[y_test[idx]]
    feat = X_test_scaled[idx:idx+1]
    
    pred_idx = best_svm.predict(feat)[0]
    pred_label = class_names[pred_idx]
    probs = best_svm.predict_proba(feat)[0]
    
    # Top 3 classes
    top3_idx = np.argsort(probs)[::-1][:3]
    top3_classes = [class_names[i].capitalize() for i in reversed(top3_idx)]
    top3_probs = [probs[i]*100 for i in reversed(top3_idx)]
    
    # Image plot
    img = Image.open(img_path).convert('RGB')
    axes[row, 0].imshow(img)
    is_correct = (pred_label == true_label)
    title_color = 'green' if is_correct else 'red'
    axes[row, 0].set_title(f"True: {true_label.capitalize()} | Pred: {pred_label.capitalize()}", fontsize=12, fontweight='bold', color=title_color)
    axes[row, 0].axis('off')
    
    # Bar plot
    bars = axes[row, 1].barh(top3_classes, top3_probs, color=['#4C72B0', '#4C72B0', '#2ca02c'])
    axes[row, 1].set_xlim(0, 100)
    axes[row, 1].set_xlabel("Probability (%)")
    axes[row, 1].set_title("Prediction Confidence Distribution")
    for i, v in enumerate(top3_probs):
        axes[row, 1].text(v + 1, i, f"{v:.1f}%", va='center', fontweight='bold')

plt.tight_layout()
plt.show()
